# 04 - Statistical Analysis
**Objective:** move beyond description: quantify associations with the tag (effect size + uncertainty), check test assumptions,
and fit an interpretable logistic regression.

**Rules:** *tagged* = 1 / *non-tagged* = 0 (definition unconfirmed); associations are not causes; p-values are never used alone;
many tests are run, so p-values are adjusted (Holm). The "no card until the final quarter" group (n=611) is handled explicitly
because it is 100% tagged in Data All and its meaning is unknown. Outputs are cleared before committing.

In [ ]:
import sys
sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score
from src.preprocessing import load_clean, ID_COL, TARGET
from src.feature_engineering import build_features, WINDOW_ALL, WINDOW_VALIDASI
from src.eda import fit_balance_cutpoints, savefig, PALETTE
from src.stats_utils import (chi_square_test, holm, risk_comparison, compare_numeric, skewness_table,
                             odds_ratio_table, lr_test, vif_table, categorical_frame, logit_design)

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 170)
sns.set_style("whitegrid"); FIG = "../figures"

clean_all = load_clean("../data/raw/finanku_data_all.csv")
clean_val = load_clean("../data/raw/finanku_data_validasi.csv")
FA, FV = build_features(clean_all, WINDOW_ALL), build_features(clean_val, WINDOW_VALIDASI)
yA, yV = clean_all.set_index(ID_COL)[TARGET], clean_val.set_index(ID_COL)[TARGET]
cuts = fit_balance_cutpoints(FA)                     # balance tiers fitted on Data All, applied to both windows
WINDOWS = {"All (Q1-Q4)": (FA, yA), "Validasi (Q2-Q5)": (FV, yV)}

## 1. Descriptive statistics by tag status

In [ ]:
cols = ["age", "income_measure", "balance_mean", "balance_change_abs", "products_mean",
        "n_zero_balance_quarters", "active_rate", "active_change"]
for name, (F, y) in WINDOWS.items():
    print(f"--- {name} ---")
    g = F[cols].groupby(y.values)
    desc = pd.concat({"mean": g.mean(), "median": g.median(), "std": g.std()}, axis=1)
    display(desc.rename(index={0: "non-tagged", 1: "tagged"}).T.round(2))

print("Skewness / normality (D'Agostino test): large samples reject normality easily, skewness shows how far from symmetric.")
display(skewness_table(FA, ["age", "income_measure", "balance_mean", "products_mean"]).round(3))

## 2. Cross-tabulation and chi-square tests
Test: chi-square test of independence between each categorical variable and the tag. **Assumption:** expected counts should be about 5 or more
(`min_expected` below). **Effect size:** Cramer's V (bias-corrected); with large n, tiny effects can still give tiny p-values, so V matters more.
Rough guide for V: <0.1 negligible, 0.1-0.3 small to moderate, >0.3 substantial. p-values are Holm-adjusted across the 8 tests in each window.

In [ ]:
chi_tables = {}
for name, (F, y) in WINDOWS.items():
    C_ = categorical_frame(F, cuts)
    rows = []
    for var in C_.columns:
        r = chi_square_test(C_[var], y); r["variable"] = var; rows.append(r)
    t = pd.DataFrame(rows).set_index("variable")
    t["p_holm"] = holm(t["p_value"])
    chi_tables[name] = t
    print(f"--- {name} ---")
    display(t[["levels", "chi2", "dof", "p_value", "p_holm", "cramers_v", "min_expected"]].round(4))

fig, ax = plt.subplots(figsize=(8, 4))
v = pd.DataFrame({k: t["cramers_v"] for k, t in chi_tables.items()}).sort_values("All (Q1-Q4)")
y_pos = np.arange(len(v)); h = 0.38
ax.barh(y_pos - h/2, v["All (Q1-Q4)"], h, color=PALETTE["All"], label="Data All (Q1-Q4)")
ax.barh(y_pos + h/2, v["Validasi (Q2-Q5)"], h, color=PALETTE["Validasi"], label="Data Validasi (Q2-Q5)")
ax.set_yticks(y_pos); ax.set_yticklabels(v.index); ax.set_xlabel("Cramer's V (association strength)")
ax.set_title("Association of categorical variables with the tag", fontweight="bold"); ax.legend(fontsize=8)
fig.tight_layout(); savefig(fig, "11a_cramers_v.png", FIG); plt.show()

## 3. Numeric variables: tagged vs non-tagged
These variables are skewed (section 1), so the **Mann-Whitney U test** is the primary test (no normality assumption); the Welch t-test is
shown for reference. Effect sizes: rank-biserial correlation (positive = tagged customers tend to have larger values) and Cohen's d with an approximate 95% CI.
Rough guide for d: 0.2 small, 0.5 medium, 0.8 large.

In [ ]:
num_tables = {}
for name, (F, y) in WINDOWS.items():
    rows = []
    for c in cols:
        r = compare_numeric(F.loc[y == 1, c], F.loc[y == 0, c]); r["variable"] = c; rows.append(r)
    t = pd.DataFrame(rows).set_index("variable")
    t["mwu_p_holm"] = holm(t["mwu_p"])
    num_tables[name] = t
    print(f"--- {name} ---")
    display(t[["median_tagged", "median_non", "mwu_p", "mwu_p_holm", "rank_biserial", "welch_p", "cohens_d", "d_low", "d_high"]].round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
for off, (name, color) in zip([-0.15, 0.15], [("All (Q1-Q4)", PALETTE["All"]), ("Validasi (Q2-Q5)", PALETTE["Validasi"])]):
    t = num_tables[name].loc[cols]
    ax.errorbar(t["cohens_d"], np.arange(len(t)) + off, xerr=[t["cohens_d"] - t["d_low"], t["d_high"] - t["cohens_d"]],
                fmt="o", color=color, capsize=3, label=name)
ax.axvline(0, color="#333", lw=0.8); ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
ax.set_xlabel("Cohen's d (tagged minus non-tagged), 95% CI"); ax.legend(fontsize=8)
ax.set_title("Standardised differences between tagged and non-tagged", fontweight="bold")
fig.tight_layout(); savefig(fig, "11b_cohens_d.png", FIG); plt.show()

## 4. Confidence intervals for group differences
Risk difference uses the Newcombe-Wilson interval; risk ratio and odds ratio use log-method intervals. If a cell is zero, the odds ratio cannot be estimated reliably (`zero_cell`).

In [ ]:
def compare_groups(F, y, mask1, mask0, label):
    r = risk_comparison(int(y[mask1].sum()), int(mask1.sum()), int(y[mask0].sum()), int(mask0.sum()))
    r["comparison"] = label; r["n_1"], r["n_0"] = int(mask1.sum()), int(mask0.sum())
    return r

rows = []
for name, (F, y) in WINDOWS.items():
    tier = pd.Series(categorical_frame(F, cuts)["balance_tier"], index=F.index).astype(str)
    prod = F["products_end"].clip(upper=4)
    for label, m1, m0 in [("No card at start vs card", F["card_start"] == 0, F["card_start"] == 1),
                          ("4+ products vs 1 product", prod == 4, prod == 1),
                          ("2 products vs 1 product", prod == 2, prod == 1),
                          ("High balance vs no balance", tier == "High", tier == "No balance")]:
        r = compare_groups(F, y, m1, m0, label); r["window"] = name; rows.append(r)
rc = pd.DataFrame(rows).set_index(["window", "comparison"])
display(rc[["n_1", "n_0", "rate_1", "rate_0", "risk_diff", "rd_low", "rd_high", "risk_ratio", "rr_low", "rr_high",
            "odds_ratio", "or_low", "or_high", "zero_cell"]].round(3))
print("Where zero_cell is True the odds ratio is not interpretable (e.g. no card in Data All: 611 of 611 tagged).")

## 5. Logistic regression (interpretable statistical model)
- **Outcome:** tag (1 = tagged). **Coefficients** are shown as **odds ratios** (OR > 1: higher odds of being tagged, holding other variables fixed).
- Product count and balance are categorical (non-linear relationship). Reference levels: balance tier *No balance*, products = 1, city = Surabaya.
- **M1:** all customers, without card information. **M2:** card holders only (removes the special no-card group).
  **M3 (Data Validasi only):** adds the card indicator. In Data All a card indicator cannot be estimated: every no-card customer is tagged (**perfect separation**).
- In-sample AUC is a *descriptive* fit statistic, **not** a prediction performance estimate (that is Phase 6-7).

In [ ]:
print("Separation check, Data All (card status x tag):")
display(pd.crosstab(categorical_frame(FA, cuts)["card_status"], yA))

def fit_logit(F, y, include_card=False, mask=None):
    if mask is not None:
        F, y = F[mask], y[mask]
    X = logit_design(F, cuts, include_card)
    return sm.Logit(y.loc[X.index], sm.add_constant(X.astype(float))).fit(disp=0, maxiter=200), X, y.loc[X.index]

models = {}
for name, (F, y) in WINDOWS.items():
    models[(name, "M1 all customers")] = fit_logit(F, y)
    models[(name, "M2 card holders only")] = fit_logit(F, y, mask=F["card_start"] == 1)
models[("Validasi (Q2-Q5)", "M3 all + card")] = fit_logit(FV, yV, include_card=True)

summary = []
for (name, label), (m, X, yy) in models.items():
    summary.append({"window": name, "model": label, "n": int(m.nobs), "tagged": int(yy.sum()), "LR_test_p": m.llr_pvalue,
                    "McFadden_R2": m.prsquared, "in_sample_AUC": roc_auc_score(yy, m.predict()), "max_VIF": vif_table(X)["VIF"].max()})
display(pd.DataFrame(summary).set_index(["window", "model"]).round(4))

In [ ]:
for (name, label), (m, X, yy) in models.items():
    print(f"=== {name} | {label} ===")
    display(odds_ratio_table(m).round(3))

In [ ]:
# Joint tests (likelihood ratio) for the groups of terms: is product count / balance tier associated with the tag overall?
rows = []
for (name, label), (m, X, yy) in models.items():
    if label.startswith("M3"):
        continue
    for grp, prefix in [("products", "products_"), ("balance tier", "balance_"), ("city", "city_")]:
        terms = [c for c in X.columns if c.startswith(prefix)]
        reduced = sm.Logit(yy, sm.add_constant(X.drop(columns=terms).astype(float))).fit(disp=0)
        r = lr_test(m, reduced); r.update({"window": name, "model": label, "terms": grp}); rows.append(r)
lr = pd.DataFrame(rows).set_index(["window", "model", "terms"])
lr["p_holm"] = holm(lr["p_value"])
display(lr.round(5))

In [ ]:
# Forest plot: odds ratios (card holders only, M2) in both windows, log scale
fig, ax = plt.subplots(figsize=(8.5, 6))
terms = [t for t in odds_ratio_table(models[("All (Q1-Q4)", "M2 card holders only")][0]).index]
for off, (name, color) in zip([-0.15, 0.15], [("All (Q1-Q4)", PALETTE["All"]), ("Validasi (Q2-Q5)", PALETTE["Validasi"])]):
    t = odds_ratio_table(models[(name, "M2 card holders only")][0]).loc[terms]
    ax.errorbar(t["odds_ratio"], np.arange(len(t)) + off, xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]],
                fmt="o", color=color, capsize=3, label=name)
ax.axvline(1, color="#333", lw=0.8); ax.set_xscale("log"); ax.set_yticks(range(len(terms))); ax.set_yticklabels(terms, fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("Odds ratio (log scale), 95% CI"); ax.legend(fontsize=8)
ax.set_title("Logistic regression, card holders only (M2)", fontweight="bold")
fig.tight_layout(); savefig(fig, "12_logit_forest_cardholders.png", FIG); plt.show()

## Interpretation notes
Summarise in `reports/statistical_findings.md` after reviewing the outputs:
1. Which variables have a statistically *and* practically meaningful association (look at effect size and CI, not only p)?
2. Do associations hold in both windows? (They are expected to be weaker in Data Validasi.)
3. Remember: these are associations in observational data with an unconfirmed target definition; they do not show that any variable causes tagging.